In [ ]:
# 超参数与路径配置
import os
from pathlib import Path
PROJECT_ROOT = Path(__file__).resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
IMAGE_SIZE = 224
NUM_WORKERS = 4
MODEL_NAME = "resnet50"
NUM_CLASSES = 10
PRETRAINED = True
FREEZE_BACKBONE = False
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 7
DEVICE = "cuda"
N_SPLITS = 5
SHUFFLE = True
RANDOM_SEED = 42


In [ ]:
# 数据集与数据加载器
import numpy as np
from torch.utils.data import Dataset,DataLoader,Subset
from torchvision import transforms
from PIL import Image
from config import IMAGE_SIZE,BATCH_SIZE,NUM_WORKERS
def get_train_transforms():
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE+32,IMAGE_SIZE+32)),
        transforms.RandomCrop(IMAGE_SIZE),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225]),
    ])
def get_val_transforms():
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225]),
    ])
class CustomImageDataset(Dataset):
    def __init__(self,image_paths,labels,transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self,idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image,label
def create_dataloaders(train_indices,val_indices,full_dataset):
    train_subset = Subset(full_dataset,train_indices)
    val_subset = Subset(full_dataset,val_indices)
    val_subset.dataset.transform = get_val_transforms()
    train_loader = DataLoader(
        train_subset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=True,
    )
    val_loader = DataLoader(
        val_subset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    return train_loader,val_loader
def load_dataset_from_dir(data_dir,transform=None):
    from torchvision.datasets import ImageFolder
    dataset = ImageFolder(root=str(data_dir),transform=transform)
    image_paths = [item[0] for item in dataset.samples]
    labels = [item[1] for item in dataset.samples]
    class_names = dataset.classes
    return image_paths,labels,class_names

In [ ]:
#模型定义（预训练 + 微调）
import torch
import torch.nn as nn
from torchvision import models
from config import MODEL_NAME,NUM_WORKERS,PRETRAINED,FREEZE_BACKBONE
def build_model():
    if MODEL_NAME == "resnet50":
        weights = models.Resnet50_Weights.DEFAULT if PRETRAINED else None
        model = models.resnet50(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features,NUM_CLASSES),
        )
    elif MODEL_NAME == 'resnet18':
        weights = models.ResNet18_Weights.DEFAULT if PRETRAINED else None
        model = models.resnet18(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features,NUM_CLASSES),
        )
    elif MODEL_NAME == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if PRETRAINED else None
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_fetures
        model.classifier[1] = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features,NUM_CLASSES),
        )
    else:
        raise ValueError(f"不支持的模型: {MODEL_NAME}")
    if FREEZE_BACKBONE:
        for param in model.parameters():
            param.requires_grad = False
        for param in model.fc.parameters():
            param.requires_grad = True
    return model
def get_optimizer(model,lr,weight_decay):
    backbone_params = []
    head_params = []
    for name,param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "fc" in name or "classifier" in name:
            head_params.append(param)
        else:
            backbone_params.append(param)
    optimizer = torch.optim.AdamW([
        {"params":backbone_params,"lr":lr*0.1},
        {"params":head_params,"lr":lr},
        ],weight_decay = weight_decay)
    return optimizer
def get_scheduler(optimizer,total_epochs):
    return torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,T_max=total_epochs,eta_min=1e-7
    )

In [ ]:
#交叉验证训练主流程
import time
import copy
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    precision_recall_curve,average_precision_score,
    precission_score,recall_score,f1_score,accuracy_score,
)
from config import (
    DATA_DIR,OUTPUT_DIR,EPOCHS,LEARNING_RATE,WEIGHT_DECAY,
    N_SPLITS,SHUFFLE,RANDOM_SEED,DEVICE,PATIENCE,
)
from dataset import (
    CustomImageDataset,create_dataloaders,
    get_train_transforms,get_val_transforms,load_dataset_from_dir,
)
from model import build_model,get_optimizer,get_scheduler
from utils import plot_training_curves,plot_precision_recall_curves
def train_one_epoch(model,dataloader,criterion,optimizer,device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images,labels in dataloader:
        images,labels = images.to(device),labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs,labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _,predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss,epoch_acc
@torch.no_grad()
def validate(model,dataloader,criterion,device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    for images,labels in dataloader:
        images,labels = images.to(device),labels.to(device)

        outputs = model(images)
        loss = criterion(outputs,labels)
        running_loss += loss.item() * images.size(0)
        _,predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss,epoch_acc,np.array(all_preds),np.array(all_labels)
class EarlyStopping:
    def __init__(self,patience=7,delta=0.0):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_state = None
    def __call__(self,val_score,model):
        score = val_score
        if self.best_score is None:
            self.best_score = score
            self.best_state = copy.deepcopy(model.state_dict())
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter = 0
        return self.best_state
def train_one_fold(fold,train_loader,val_Loader,device):
    model = build_model().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = get_optimizer(model,lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    scheduler = get_scheduler(optimizer,total_epochs=EPOCHS)
    early_stopping = EarlyStopping(patience=PATIENCE)
    history = {
        "train_loss":[],"train_acc":[],
        "val_loss":[],"val_acc":[],
    }
    print(f"\n{'='*60}")
    print(f"Fold{fold+1}")
    print(f"{'='*60}")
    for epoch in range(EPOCHS):
        t_start = time.time()
        train_loss,train_acc = train_one_epoch(
            model,train_loader,criterion,optimizer,device
        )
        val_loss,val_acc, _, _ = validate(
            model,val_Loader,criterion,device
        )
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        scheduler.step()
        current_lr = optimizer.param_groups[-1]["lr"]
        best_state = early_stopping(val_loss,model)
        if early_stopping.early_stop:
            print(f"  Epoch {epoch+1:3d} | Early Stopping triggered.")
            break
        elapsed = time.time() - t_start
        print(
            f"  Epoch {epoch+1:3d}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f} | "
            f"LR: {current_lr:.2e} | {elapsed:.1f}s"
        )
        model.load_state_dict(best_state)
        return model,history
def main():
    device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")
    print(f"使用设备: {device}")
    image_paths,labels,class_names = load_dataset_from_dir(DATA_DIR)
    labels_array = np.array(labels)
    print(f"数据集大小: {len(image_paths)} 张图像, {len(class_names)} 个类别")
    print(f"类别: {class_names}")
    full_dataset = CustomImageDataset(
        image_paths=image_paths,
        labels=labels,
        transform=get_train_transforms(),
    )
    skf = StratifiedKFold(n_splits=N_SPLITS,shuffle=SHUFFLE,random_state=RANDOM_SEED)
    fold_metrics = []
    for fold,(train_indices,val_indices) in enumerate(skf.split(image_paths,labels_array)):
        print(f"\n训练集样本数: {len(train_indices)} | 验证集样本数: {len(val_indices)}")
        train_loader,val_loader = create_dataloaders(
            train_indices,val_indices,full_dataset
        )
        model,history=train_one_fold(fold,train_loader,val_loader,device)
        plot_training_curves(
            history,
            save_path=OUTPUT_DIR / f"fold_{fold+1}_training_curves.png",
            fold=fold+1,
        )
        val_loss,val_acc,all_preds,all_labels = validate(
            model,val_loader,nn.CrossEntropyLoss(),device
        )
        precision = precission_score(all_labels,all_preds,average='macro',ero_division=0)
        recall = recall_score(all_labels,all_preds,average="macro",zero_division=0)
        f1 = f1_score(all_labels,all_preds,average="macro",zero_division=0)
        accuracy = accuracy_score(all_labels,all_preds)
        print(f"\n  Fold {fold+1} 最终评估:")
        print(f"    Accuracy:  {accuracy:.4f}")
        print(f"    Precision: {precision:.4f}")
        print(f"    Recall:    {recall:.4f}")
        print(f"    F1-Score:  {f1:.4f}")
        fold_metrics.append({
            "accuracy":accuracy,
            "precision":precision,
            "recall":recall,
            "f1":f1,
        })
        plot_pr_curves_for_fold(model,val_loader,class_names,fold,device) # type: ignore
